In [13]:
import sys
import os

path_to_scripts = os.path.join('..', '..', 'python_scripts')
sys.path.append(path_to_scripts)

%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_error, max_error

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

from data_to_bigquery import load_from_bigquery
from feature_engineering import drop_lag_nulls, validate_features, engineer_features
from baseline_model import baseline_model_xgb, xgb_training_preproc


%matplotlib inline

ImportError: cannot import name 'xgb_training_preproc' from 'baseline_model' (/Users/madeleine/code/mlbh/GridZero/notebooks/data/../../python_scripts/baseline_model.py)

In [ ]:
# BBoujie af LLM CHECK to help debugging to make sure all tables parse
# tables = ['test_merge_2017_onward_raw', 'test_merge_with_the_sauce']

# for table in tables:
#     print(f"\n--- TESTING PREPROC: {table} ---")

#     df = load_from_bigquery(
#         PROJECT='gridzero-489711',
#         DATASET='merged_set',
#         TABLE=table
#     )

#     df_pre = baseline_preproc(df)

#     print("raw shape:", df.shape)
#     print("preproc shape:", df_pre.shape)
#     print("columns after preproc:", df_pre.columns.tolist())

#     target_candidates = [c for c in df_pre.columns if 'carbon_intensity' in c]
#     print("target candidates:", target_candidates)

#     if target_candidates:
#         target_col = target_candidates[0]
#         print("target NA count:", df_pre[target_col].isna().sum())

#     lag_cols = [c for c in ['lag_48', 'lag_336', 'lag_17520'] if c in df_pre.columns]
#     print("lag cols present:", lag_cols)

#     if lag_cols:
#         print("lag NA counts:")
#         print(df_pre[lag_cols].isna().sum())

#     print("total NA count in preproc df:", df_pre.isna().sum().sum())

In [ ]:
# BBoujie af LLM CHECK to help debugging to make sure all tables parse
for table in tables:
    print(f"\n--- TESTING X / y: {table} ---")

    df = load_from_bigquery(
        PROJECT='gridzero-489711',
        DATASET='merged_set',
        TABLE=table
    )

    df_pre = baseline_preproc(df)

    target_col = [c for c in df_pre.columns if 'carbon_intensity' in c][0]

    X = df_pre.drop(columns=[target_col, 'datetime'], errors='ignore').copy()
    X = X.select_dtypes(include='number').astype(float)

    y = df_pre[target_col].astype(float)

    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("non-numeric cols left:", X.select_dtypes(exclude='number').columns.tolist())
    print("X total NA count:", X.isna().sum().sum())
    print("y total NA count:", y.isna().sum())

In [ ]:
# BBoujie af LLM CHECK to help debugging to make sure all tables parse
for table in tables:
    print(f"\n--- FITTING MODEL: {table} ---")
    model = baseline_model_xgb(TABLE=table)
    print(model)

In [9]:
# raw table

df_raw = load_from_bigquery('gridzero-489711', 'merged_set', 'test_merge_2017_onward_raw')
display(df_raw.head())
df_raw.columns


/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 148,991 rows and 26 columns from gridzero-489711.merged_set.test_merge_2017_onward_raw


,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,...,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,TotalOutput-MW,carbon_intensity_gCO2_kWh,status
0,2017-09-12 00:00:00,11.6,31.0,28.1,4,0.0,0.0,0.0,1001.2,0.0,...,0.0,265.0,7897.0,173.0,0.0,4006.573,4394.973,21722.546,142.0,okay
1,2017-09-12 00:30:00,11.6,31.0,28.1,4,0.0,0.0,0.0,1001.2,0.0,...,0.0,248.0,7897.0,174.0,0.0,3973.985,4418.446,21554.431,140.0,okay
2,2017-09-12 01:00:00,11.2,30.3,27.0,5,0.0,0.0,0.0,1001.9,0.0,...,0.0,242.0,7852.0,171.0,0.0,3941.698,4533.019,21767.717,139.0,okay
3,2017-09-12 01:30:00,11.2,30.3,27.0,5,0.0,0.0,0.0,1001.9,0.0,...,0.0,242.0,7706.0,174.0,0.0,3945.620,4726.191,21742.811,137.0,okay
4,2017-09-12 02:00:00,10.9,29.6,25.2,7,0.0,0.0,0.0,1002.4,0.0,...,0.0,242.0,7684.0,173.0,0.0,3919.454,4832.410,21579.864,132.0,okay


Index(['datetime', 'temperature_2m_c', 'wind_speed_100m_ms',
       'wind_gusts_10m_ms', 'cloud_cover_pct', 'shortwave_radiation_wm2',
       'direct_radiation_wm2', 'diffuse_radiation_wm2', 'pressure_msl_hpa',
       'snowfall_cm', 'rain_mm', 'precipitation_mm', 'Biomass', 'Fossil Gas',
       'Fossil Hard coal', 'Fossil Oil', 'Hydro Pumped Storage',
       'Hydro Run-of-river and poundage', 'Nuclear', 'Other', 'Solar',
       'Wind Offshore', 'Wind Onshore', 'TotalOutput-MW',
       'carbon_intensity_gCO2_kWh', 'status'],
      dtype='object')

In [ ]:
# Joe's saucey table
df_saucey = load_from_bigquery('gridzero-489711', 'merged_set', 'test_merge_with_the_sauce')

print(df.shape)
print(df.dtypes)
print(df.isna().sum().sort_values(ascending=False).head(20))

In [ ]:
df = baseline_preproc(df, target_col='carbon_intensity_gCO2_kWh')

lag_cols = [col for col in ['lag_48', 'lag_336', 'lag_17520'] if col in df.columns]
if lag_cols:
    df = df.dropna(subset=lag_cols)

X = df.drop(columns=['carbon_intensity_gCO2_kWh', 'datetime'], errors='ignore')

print(X.shape)
print(X.dtypes)
print(X.isna().sum().sort_values(ascending=False).head(20))
print(X.select_dtypes(exclude=['number', 'bool']).columns.tolist())

In [ ]:
df_saucey = load_from_bigquery('gridzero-489711', 'merged_set', 'test_merge_with_the_sauce')
display(df_saucey.head())

print(df_saucey.columns.tolist())
print(df_saucey.dtypes)


In [ ]:
# check carb intensity from engingeer features function
# df = engineer_features(df, target_col='carbon_intensity_gCO2_kWh')
# df.head()

In [ ]:
# debugging python imports
# import baseline_model
# print(baseline_model.__file__)
# print('baseline_preproc' in dir(baseline_model))
# print(dir(baseline_model))

In [ ]:
# testing preproc
baseline_preproc(df)

In [ ]:
model = baseline_model_xgb('gridzero-489711', 'merged_set', 'Fully_merged_dataset_2025')
model

In [ ]:
# def load_from_bigquery(PROJECT: str, DATASET: str, TABLE: str):
#     '''
#     Load data from BigQuery into a pandas DataFrame.

#     Arguments:
#         PROJECT: GCP project ID
#         DATASET: BigQuery dataset name
#         TABLE: BigQuery table name

#     Returns: pandas DataFrame
#     '''

#     client = bigquery.Client(project=PROJECT)

#     query = f'''
#     SELECT *
#     FROM `{PROJECT}.{DATASET}.{TABLE}`
#     '''

#     df = client.query(query).to_dataframe()

#     rows, cols = df.shape
#     print(f'Loaded {rows:,} rows and {cols} columns from {PROJECT}.{DATASET}.{TABLE}')

#     return df

In [ ]:
df = load_from_bigquery('gridzero-489711', 'merged_set', 'feature_engineered_dataset_2025')

In [ ]:
df.head(1)
df.dtypes

In [ ]:
# model fucntion checking and development
def baseline_model_xgb(PROJECT: str='gridzero-489711',
                       DATASET: str='merged_set',
                       TABLE: str='feature_engineered_dataset_2025',
                       test_size:int = 0.3):

    # pull df from BQ
    df = load_from_bigquery(PROJECT=PROJECT, DATASET=DATASET, TABLE=TABLE)

    # set features and target variables
    X = df.drop(columns=['carbon_intensity'])
    y = df['carbon_intensity']

    # test train split for single year
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size)

    #put column transformer into pipeline to ignore the columns
    cols_to_keep_X = [c for c in df.columns if c not in ['datetime']]

    col_tf = ColumnTransformer(transformers=[
                ('choose_cols', 'passthrough', cols_to_keep_X)
        ])

    # build simple xgb model
    pipeline = make_pipeline(
                    col_tf,
                    StandardScaler(),
                    XGBRegressor()
                 )

    # training model
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test.iloc[:5])
    print(y_pred)
    return pipeline

test_model = baseline_model_xgb()
test_model


In [ ]:
print(f'Shape: {df.shape}')
print(f'Nulls:\n{df.isnull().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')

In [ ]:
df['Fossil Gas']

In [ ]:
cols = [df.columns]
cols

In [ ]:

df.plot(x='time', y='carbon_intensity', figsize=(15,4))
plt.title('Carbon Intensity 2025')
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm')
plt.title('Feature Correlations')
plt.show()

In [ ]:
feat_corr_matrix_plot, ax = plt.subplots()

corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm', ax = ax)
plt.title('Feature Correlations')
plt.show()

In [ ]:
feat_corr_matrix_plot

In [ ]:
load_from_bigquery('gridzero-489711', 'merged_set', 'Fully_merged_dataset_2025')

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 17,519 rows and 23 columns from gridzero-489711.merged_set.Fully_merged_dataset_2025


,time,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,...,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,TotalOutput-MW,carbon_intensity
0,2025-01-01 00:00:00+00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,...,0.0,0.0,736.0,5065.0,183.0,0.0,11444.531,9113.028,31028.559,55
1,2025-01-01 00:30:00+00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,...,0.0,0.0,745.0,5063.0,290.0,0.0,11138.565,8969.868,31138.433,54
2,2025-01-01 01:00:00+00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,...,0.0,0.0,746.0,5056.0,333.0,0.0,10788.770,8931.922,30826.692,53
3,2025-01-01 01:30:00+00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,...,0.0,0.0,746.0,5060.0,238.0,0.0,10519.534,8810.976,30209.510,53
4,2025-01-01 02:00:00+00:00,11.1,43.6,60.5,100,0.0,0.0,0.0,1012.0,0.0,...,0.0,0.0,747.0,5056.0,277.0,0.0,10706.056,8456.386,29988.442,47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17514,2025-12-31 21:00:00+00:00,-0.8,22.5,23.4,5,0.0,0.0,0.0,1023.1,0.0,...,0.0,4.0,378.0,4082.0,743.0,0.0,10803.097,5827.825,31071.922,<NA>
17515,2025-12-31 21:30:00+00:00,-0.8,22.5,23.4,5,0.0,0.0,0.0,1023.1,0.0,...,0.0,4.0,376.0,4086.0,669.0,0.0,10832.486,5820.693,30313.179,<NA>
17516,2025-12-31 22:00:00+00:00,-0.9,23.3,23.8,0,0.0,0.0,0.0,1021.9,0.0,...,0.0,4.0,361.0,4097.0,494.0,0.0,10937.104,6065.013,30201.117,<NA>
17517,2025-12-31 22:30:00+00:00,-0.9,23.3,23.8,0,0.0,0.0,0.0,1021.9,0.0,...,0.0,4.0,353.0,4094.0,253.0,0.0,11018.705,6001.891,30088.596,<NA>


In [ ]:
df.rename(columns={'time': 'datetime'})
df = df.rename(columns={'time': 'datetime'})

In [ ]:
display(df.head(5))

display(engineer_features(df, target_col='carbon_intensity'))

In [ ]:
df = engineer_features(df)
validate_features(df)
df = drop_lag_nulls(df)
df

In [ ]:
# data features and target
# note dropped target AND datime
# note renamed time to datetime to make sure engineerign features work
X = df.drop(columns=['carbon_intensity', 'datetime'])
y = df['carbon_intensity']

print(X.dtypes)

In [ ]:
y


In [ ]:
# test train split for 2025
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.30)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
# test train split for more years
# df = df.sort_values('datetime').copy()

# split_date = '2025-10-01'

# train_df = df[df['datetime'] < split_date]
# test_df = df[df['datetime'] >= split_date]

# X_train = train_df.drop(columns=['carbon_intensity', 'datetime'])
# y_train = train_df['carbon_intensity']

# X_test = test_df.drop(columns=['carbon_intensity', 'datetime'])
# y_test = test_df['carbon_intensity']

In [ ]:
# simple pipelines

# pipeline = make_pipeline(
#     XGBRegressor(
#         n_estimators=100,
#         learning_rate=0.1,
#         max_depth=6,
#     )
# )

# pipeline_xgbr1 = make_pipeline(
#     StandardScaler(),
#     XGBRegressor(
#         n_estimators=100,
#         learning_rate=0.1,
#         max_depth=6,
#         random_state=42
#     )
# )


In [ ]:
# pipeline.fit(X_train, y_train)
# y_pred = pipeline.predict(X_test)
# print(y)
# print(y_pred)

In [ ]:
# results = cross_validate(
#     pipeline,
#     X_train,
#     y_train,
#     cv=5,
#     scoring=['neg_mean_squared_error', 'neg_root_mean_squared_error', 'r2', 'neg_mean_absolute_error', 'neg_max_error']
# )

# results


In [ ]:
# test_results = {k: v for k, v in results.items() if k.startswith('test')}
# test_results = pd.DataFrame(test_results)

# display(test_results)

In [ ]:
# plt.figure(figsize=(10,4))
# plt.plot(y_test.values, label='true')
# plt.plot(y_pred, label='pred')
# plt.show()

In [ ]:
# DO NOT NEED bc we have enegineered features
# function to extract from datetime
# def extract_datetime_features(df):
#     df = df.copy()
#     dt = pd.to_datetime(df.iloc[:, 0])

#     return pd.DataFrame({
#         'hour': dt.dt.hour,
#         'day': dt.dt.day,
#         'month': dt.dt.month,
#         'dayofweek': dt.dt.dayofweek
#     }, index=df.index)

In [ ]:
# model comparison
models = {
    'dummy': make_pipeline(
        DummyRegressor(strategy='mean')
    ),
    'linear_regression': make_pipeline(
        StandardScaler(),
        LinearRegression()
    ),
    'random_forest': make_pipeline(
        RandomForestRegressor(
            n_estimators=100,
            random_state=42
        )
    ),
    'xgboost_default': make_pipeline(
            XGBRegressor()
    ),

    'xgboost': make_pipeline(
        XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=6,
            random_state=42,
            objective='reg:squarederror'
        )
    ),
    'xgboost_scl': make_pipeline(
        StandardScaler(),
        XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=6,
            random_state=42
        )
    ),
    'xgboost_opt' : make_pipeline(
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=5,
            #L1
            reg_alpha=0.1,
            #L2
            reg_lambda=0,
            # histo method (faster than default greedy algo)
            tree_method='hist',
            random_state=42
            )
    ),

    'xgboost_opt_scl' : make_pipeline(
        StandardScaler(),
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=5,
            reg_alpha=0.1,
            reg_lambda=1.0,
            tree_method='hist',
            random_state=42
            )

    )
}

In [ ]:
# scoring
scoring = {
    'mae': 'neg_mean_absolute_error',
    'rmse': 'neg_root_mean_squared_error',
    'r2': 'r2',
    'max_err':'neg_max_error'
}

In [ ]:
# loop through models
# results
results = []

for model_name, pipeline in models.items():
    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        return_train_score=False
    )

    model_results = {
        'model': model_name,
        'model_fit_t': cv_results['fit_time'].mean(),
        'mae_mean': -cv_results['test_mae'].mean(),
        'rmse_mean': -cv_results['test_rmse'].mean(),
        'r2_mean': cv_results['test_r2'].mean(),
        'max_err_max': -cv_results['test_max_err'].max(),
    }

    results.append(model_results)

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('mae_mean')

display(results_df)

In [ ]:
# results:
# XGBOOST low MAE/RMSE/r2

In [ ]:
# simple model and predict in funcion
# pipeline = pipeline = make_pipeline(
#         StandardScaler(),
#         XGBRegressor()
# )

# pipeline.fit(X_train, y_train)

# y_pred = pipeline.predict(X_test)
# y_pred


In [ ]:
# testing fucntion codes

# backend babyyy
model, feature_cols = baseline_model_xgb(TABLE='test_merge_with_the_sauce')

# simulating a front end table
df_fr = load_from_bigquery('gridzero-489711','merged_set','test_merge_with_the_sauce')
print(df_fr)

# testing
X_new = prepare_prediction_features(df_fr, feature_cols)

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 1,535 rows and 27 columns from gridzero-489711.merged_set.test_merge_with_the_sauce


/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 1,535 rows and 27 columns from gridzero-489711.merged_set.test_merge_with_the_sauce


,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,...,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,TotalOutput-MW,carbon_intensity_gCO2_kWh,hour,month
0,2023-01-01 00:00:00,11.2,38.8,63.7,100,0.0,0.0,0.0,1002.8,0.0,...,0.0,0.0,0.0,0.0,6135.092,4783.192,10918.284,72.0,0.0,1.0
1,2023-01-01 00:30:00,11.2,38.8,63.7,100,0.0,0.0,0.0,1002.8,0.0,...,0.0,0.0,0.0,0.0,6269.940,4781.652,11051.592,80.0,0.0,1.0
2,2023-01-01 01:00:00,11.9,49.2,54.7,100,0.0,0.0,0.0,1003.6,0.0,...,0.0,0.0,0.0,0.0,6434.796,4603.496,11038.292,72.0,1.0,1.0
3,2023-01-01 01:30:00,11.9,49.2,54.7,100,0.0,0.0,0.0,1003.6,0.0,...,0.0,0.0,0.0,0.0,6724.632,4385.486,11110.118,65.0,1.0,1.0
4,2023-01-01 02:00:00,11.0,50.6,51.5,100,0.0,0.0,0.0,1004.6,0.0,...,0.0,0.0,0.0,0.0,6946.517,4024.265,10970.782,65.0,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1530,2023-02-01 21:00:00,6.5,36.4,33.5,96,0.0,0.0,0.0,1025.8,0.0,...,611.0,4800.0,212.0,0.0,200.488,2374.957,16383.445,NaN,21.0,2.0
1531,2023-02-01 21:30:00,6.5,36.4,33.5,96,0.0,0.0,0.0,1025.8,0.0,...,524.0,4805.0,218.0,0.0,6898.209,5549.160,24210.369,NaN,21.0,2.0
1532,2023-02-01 22:00:00,6.3,36.7,32.0,61,0.0,0.0,0.0,1026.1,0.0,...,581.0,4804.0,175.0,0.0,6886.923,5617.587,23353.510,NaN,22.0,2.0
1533,2023-02-01 22:30:00,6.3,36.7,32.0,61,0.0,0.0,0.0,1026.1,0.0,...,456.0,4810.0,180.0,0.0,6614.979,5540.601,22129.580,NaN,22.0,2.0


In [ ]:
# pipeline to with added params

# pipeline = pipeline = make_pipeline(
#         StandardScaler(),
#         XGBRegressor(
#             n_estimators=2000,
#             learning_rate=0.03,
#             max_depth=5,
#             subsample=0.8,
#             colsample_bytree=0.8,
#             reg_alpha=0.1,
#             reg_lambda=1.0,
#             #histo
#             tree_method='hist',
#             random_state=42
#             )
#         )

In [ ]:
# gridsearch params

param_grid = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

# params = {
#             'n_estimators': np.arange(200, 1000, 100),
#             'max_depth': np.arange(3, 11, 1),
#             'learning_rate': np.arange(0.01, 0.3, 0.01),
#             'subsample': [0.8, 1.0],
#             'colsample_bytree': [0.8, 1.0]
# }


In [ ]:
# grissearch cross val

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
    verbose=2
)

# grid.fit(X_train, y_train)

In [ ]:
# # best model
# best_model = grid.best_estimator_

In [ ]:
# predicting
# y_pred = pipeline.predict(X_test)

In [ ]:
# fit with girdsearch cv
